# Notebook 01 — Data Collection

Assembles raw data and writes three CSVs that Notebook 02 consumes:
- `data/raw/grants_raw.csv` — individual grant records (funder → recipient, amount, purpose)
- `data/raw/funders_raw.csv` — funder (foundation) summaries
- `data/raw/recipients_raw.csv` — unique grant recipients (name, city, state)

**Sources**
1. **IRS 990-PF index CSVs** (`apps.irs.gov`) — fast population counts of e-filed 990-PFs
2. **IRS 990-PF XML bulk ZIP** (`apps.irs.gov`) — the actual grant detail (one ~400 MB chunk)
3. **ProPublica Nonprofit Explorer API** — optional funder enrichment

> **Notes on the IRS data**
> - The IRS deprecated its AWS S3 e-file bucket in Dec 2021; data now comes as bulk ZIP
>   archives that mix 990 / 990-EZ / 990-PF, so we filter to 990-PF after download.
> - 990-PF grant records list recipient **name + address but no EIN** (the form doesn't
>   require it), so recipients are keyed by name + state. EIN-based enrichment of recipients
>   (NTEE, demographics) is left to the Candid integration.

All raw outputs are gitignored. **Run top to bottom**, then go to Notebook 02.

In [1]:
import sys
sys.path.insert(0, '..')

import time
import logging
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from src.api_client import (
    fetch_990_index,
    sample_990pf_from_zip,
    parse_990pf_grants,
    parse_990pf_funder_summary,
    propublica_organization,
)

logging.basicConfig(level=logging.INFO)
RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1  IRS 990-PF Index (fast population counts)

The index CSVs are small and list every e-filed 990-PF for a year. We use them to show the
size of the filer population by year. (Grant *detail* comes from the XML in section 2.)

In [2]:
YEARS = [2018, 2019, 2020, 2021]   # index years to summarize

counts = {}
for year in YEARS:
    counts[year] = len(fetch_990_index(year))
    print(f"{year}: {counts[year]:,} 990-PF filings")

pd.Series(counts, name='filings').to_csv(RAW / '990pf_index_counts.csv')
print('\nSaved filer population counts.')

INFO: Fetching IRS index for 2018…


INFO:   Found 67520 990-PF filings for 2018


INFO: Fetching IRS index for 2019…


2018: 67,520 990-PF filings


INFO:   Found 64614 990-PF filings for 2019


...

INFO:   Found 27883 990-PF filings for 2020


INFO: Fetching IRS index for 2021…


2020: 27,883 990-PF filings


INFO:   Found 108644 990-PF filings for 2021


2021: 108,644 990-PF filings

Saved filer population counts.


## 2  Download & parse 990-PF XML grant data

Downloads bulk ZIP archives (cached after first download) from **multiple IRS release years**
so the sample spans 2020, filters each to 990-PF returns, and parses their grant schedules.
`SAMPLE_PER_ARCHIVE` caps how many 990-PF filings we take from each archive.

Produces `grants_raw.csv` and `funders_raw.csv`.

In [3]:
# Pull from multiple IRS release archives so the sample spans 2020.
# Each release holds fiscal years roughly 1-2 years earlier:
#   2020 release -> FY ~2018-2019   (download990xml_2020_1.zip, ~400 MB)
#   2023 release -> FY ~2020-2022   (2023_TEOS_XML_01A.zip,     ~120 MB)
# Both are cached after first download. Add more (year, chunk) pairs for depth.
ARCHIVES = [(2020, '1'), (2023, '01A')]
SAMPLE_PER_ARCHIVE = 3000   # number of 990-PF filings to take from each archive

filings = {}
for yr, ch in ARCHIVES:
    filings.update(sample_990pf_from_zip(yr, chunk=ch, max_filings=SAMPLE_PER_ARCHIVE, save=True))

print(f"Collected {len(filings):,} 990-PF filings across {len(ARCHIVES)} release archives")

INFO: Using cached ZIP _chunk_2020_1.zip


INFO: Collected 3000 990-PF filings from 2020 chunk 1


INFO: Using cached ZIP _chunk_2023_01A.zip


INFO: Collected 2508 990-PF filings from 2023 chunk 01A


Collected 5,508 990-PF filings across 2 release archives


In [4]:
grants_records = []
funder_records = []

for oid, xml_text in tqdm(filings.items(), desc='Parsing 990-PF XML'):
    try:
        gs = parse_990pf_grants(xml_text)
        for g in gs:
            g['object_id'] = oid
        grants_records.extend(gs)
        funder_records.append(parse_990pf_funder_summary(xml_text))
    except Exception as e:
        print(f"parse error {oid}: {e}")

grants_raw = pd.DataFrame(grants_records)
funders_raw = pd.DataFrame(funder_records)
grants_raw.to_csv(RAW / 'grants_raw.csv', index=False)
funders_raw.to_csv(RAW / 'funders_raw.csv', index=False)

print(f"grants_raw:  {grants_raw.shape[0]:,} grant records")
print(f"funders_raw: {funders_raw.shape[0]:,} foundations")
grants_raw.head(100)

Parsing 990-PF XML:   0%|          | 0/5508 [00:00<?, ?it/s]

grants_raw:  57,277 grant records
funders_raw: 5,508 foundations


,funder_ein,funder_name,tax_year,recipient_name,recipient_ein,recipient_city,recipient_state,grant_amount,grant_purpose,object_id
0,134147704,THE MENEZES FOUNDATION INC,2018,SEE ATTACHED SCHEDULE,None,NEW YORK,NY,269500,UNRESTRICTED,202000109349100110
1,113457122,Elizabeth T McNamee Memorial,2018,WEST ISLIP ASSN SCHOOL ADMIN,None,WEST ISLIP,NY,3000,EDUCATION,202000119349100000
2,113457122,Elizabeth T McNamee Memorial,2018,ST PATRICKS SCHOLARSHIP FUND,None,BAY SHORE,NY,1000,EDUCATION,202000119349100000
3,113457122,Elizabeth T McNamee Memorial,2018,WEST ISLIP BREAST CANCER,None,WEST ISLIP,NY,750,RESEARCH,202000119349100000
4,113457122,Elizabeth T McNamee Memorial,2018,ST JOHN THE BAPTIST HS,None,WEST ISLIP,NY,1000,TUITION ASSISTANCE,202000119349100000
...,...,...,...,...,...,...,...,...,...,...
95,824452306,MARGARET AND DAVID PERRY FOUNDATION,2019,ACADEMY PREP CENTER OF ST PETERSBURG,None,ST PETERSBURG,FL,2500,FOR CHILDREN EDUCATION,202000279349100100
96,222141748,LOUIS J & FANNIE RONCOLI FOUNDATION,2019,ST HELEN'S OUTREACH PROGRAM,None,WESTFIELD,NJ,4500,OUTREACH PROGRAMS FOR THE NEEDY,202000279349100600
97,222141748,LOUIS J & FANNIE RONCOLI FOUNDATION,2019,CATHOLIC CHARITIES OF THE,None,EAST ORANGE,NJ,80000,ONE-TO-ONE NEEDY FAMILY SUPPORT,202000279349100600
98,222141748,LOUIS J & FANNIE RONCOLI FOUNDATION,2019,ROSEPETALS INC,None,BLOOMFIELD,NJ,1000,ASSISTANCE TO TERMINALLY ILL CHILDRE,202000279349100600


## 3  Unique recipients

990-PF filings give recipient name + city + state (no EIN), so we build a recipient table
keyed by normalized name + state. Produces `recipients_raw.csv`.

In [5]:
if grants_raw.empty:
    raise RuntimeError('grants_raw is empty — re-run section 2 first.')

recip = (
    grants_raw[['recipient_name', 'recipient_city', 'recipient_state']]
    .rename(columns={'recipient_name': 'name', 'recipient_city': 'city', 'recipient_state': 'state'})
    .dropna(subset=['name'])
    .drop_duplicates(['name', 'state'])
    .reset_index(drop=True)
)
# Columns kept for schema compatibility with Notebook 02 (filled via Candid later)
recip['ein'] = None
recip['ntee_code'] = None
recip['total_revenue'] = None
recip.to_csv(RAW / 'recipients_raw.csv', index=False)

print(f"recipients_raw: {len(recip):,} unique recipients")
recip.head()

recipients_raw: 45,790 unique recipients


,name,city,state,ein,ntee_code,total_revenue
0,SEE ATTACHED SCHEDULE,NEW YORK,NY,None,None,None
1,WEST ISLIP ASSN SCHOOL ADMIN,WEST ISLIP,NY,None,None,None
2,ST PATRICKS SCHOLARSHIP FUND,BAY SHORE,NY,None,None,None
3,WEST ISLIP BREAST CANCER,WEST ISLIP,NY,None,None,None
4,ST JOHN THE BAPTIST HS,WEST ISLIP,NY,None,None,None


## 4  ProPublica funder enrichment (optional)

Demonstrates the ProPublica API client by pulling profiles for a few well-known foundations.
Not required by downstream notebooks.

In [6]:
SEED_EINS = {
    '131684331': 'Ford Foundation',
    '237093598': 'John D. & Catherine T. MacArthur Foundation',
    '381359264': 'W.K. Kellogg Foundation',
    '131659629': 'Rockefeller Foundation',
    '521951681': 'Annie E. Casey Foundation',
}
rows = []
for ein, label in SEED_EINS.items():
    try:
        org = propublica_organization(ein).get('organization', {})
        rows.append({'ein': ein, 'name': org.get('name'), 'state': org.get('state'),
                     'ntee_code': org.get('ntee_code'), 'revenue': org.get('revenue_amount')})
        time.sleep(0.3)
    except Exception as e:
        rows.append({'ein': ein, 'name': label, 'error': str(e)})

pd.DataFrame(rows).to_csv(RAW / 'foundations_propublica.csv', index=False)
pd.DataFrame(rows)

,ein,name,state,ntee_code,revenue
0,131684331,Ford Foundation,NY,T21,None
1,237093598,John D & Catherine T Macarthur Foundation,IL,T20Z,None
2,381359264,W K Kellogg Foundation,MI,T21,None
3,131659629,Rockefeller Foundation,NY,NaN,None
4,521951681,Annie E Casey Foundation,MD,T21,None


## 5  Candid APIs (stub)

Activates once `CANDID_API_KEY` is set in a `.env` file at the project root.
Candid would supply recipient EINs, NTEE codes, and leadership demographics.

In [7]:
# from dotenv import load_dotenv
# load_dotenv('../.env')
# from src.api_client import candid_demographics
print('Candid API stub — set CANDID_API_KEY in .env to activate')

Candid API stub — set CANDID_API_KEY in .env to activate


## Summary

Written to `data/raw/`: `990pf_index_counts.csv`, `grants_raw.csv`, `funders_raw.csv`,
`recipients_raw.csv`, `foundations_propublica.csv`.

Proceed to **Notebook 02** for cleaning and SQLite loading.